# Agentic Components Feature Showcase

This notebook walks through the core features of the Agentic Components framework and ends with a complex, production-style use case. It is heavily annotated so you can adapt it quickly.

### What you'll see
- Lego-style agents (`AgentBuilder`) with capabilities (LLM, tools, memory, prompts, rate limits).
- Prompt management (Prompt-Framework optional) with versioning.
- with_steps (batch/parallel/conditional) and GraphRunner (DAG with dynamic additions).
- CommunicationHub (direct/broadcast/namespace) + rate limits and logging/metrics.
- Memory options (scratch, episodic, semantic, vector) with an in-notebook embedding + retrieval.
- Safe tools (sanitized code exec, safe file read).
- Run persistence (RunStore) and structured logging hooks.
- A complex "Release Readiness" use case tying everything together.

In [ ]:
# Runtime setup: optional OpenAI key with fallback stub LLM
import os
from getpass import getpass

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter OPENAI_API_KEY (press Enter to skip): ')

from agentic_codex import (
    AgentBuilder, Context, FunctionAdapter, EnvOpenAIAdapter, EnvOpenAIEmbeddingAdapter,
    CommunicationHub, CommEnvelope, PromptManager,
    MultiLimiter, RateLimiter, SafeFileReaderToolAdapter, SanitizedCodeExecutionToolAdapter,
    HashEmbeddingAdapter, VectorMemory, RunStore, StructuredLogger,
    StepSpec, run_steps, GraphRunner, GraphNodeSpec, apply_rate_limits, load_prompts
)
from agentic_codex.core.schemas import AgentStep, Message

def stub_llm(prompt: str) -> str:
    lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
    return '[stub llm] ' + (lines[-1][:200] if lines else 'response')

try:
    llm = EnvOpenAIAdapter(model='gpt-4o-mini')
except Exception:
    llm = FunctionAdapter(stub_llm)


## Prompt management (versioned prompts)
- Uses Prompt-Framework if installed (`pip install agentic-codex[prompt]`).
- Falls back to a simple dict if the optional dependency is missing.

In [ ]:
prompt_manager = PromptManager()
prompt_store = {}
try:
    prompt_manager.add('review_prompt', 'Review code and score 0-10; respond JSON.', version='v1')
    prompt_manager.add('doc_prompt', 'Document the changes and risks.', version='v1')
    use_prompt = lambda name: prompt_manager.get(name)
except Exception:
    prompt_store['review_prompt'] = 'Review code and score 0-10; respond JSON.'
    prompt_store['doc_prompt'] = 'Document the changes and risks.'
    use_prompt = lambda name: prompt_store.get(name, '')


## Memory, tools, comms, limits, logging
- VectorMemory with hash embeddings for local retrieval.
- Safe tools: sanitized code exec + safe file reader.
- Communication hub with namespace rate limits.
- RunStore for persistence (JSON); StructuredLogger for JSON logs.

In [ ]:
embedder = HashEmbeddingAdapter(dimension=64)
vec_mem = VectorMemory(embedder.embed)
vec_mem.add('calc_requirements', 'Calculator must support add/subtract/multiply/divide with CLI entry point')

safe_reader = SafeFileReaderToolAdapter(root='.')
safe_exec = SanitizedCodeExecutionToolAdapter()

hub = CommunicationHub()
hub.create_namespace('engineering', ['coder', 'reviewer', 'recommender', 'doc'])
hub.set_namespace_rate_limit('engineering', rate=5, capacity=5)

limiter = MultiLimiter()
limiter.set_limit('tool', safe_reader.name, RateLimiter(rate_per_sec=5, capacity=5))
limiter.set_limit('tool', safe_exec.name, RateLimiter(rate_per_sec=3, capacity=3))

store = RunStore('runs/')
logger = StructuredLogger(name='agentic-demo')


## Agents for complex release-readiness use case
- Coder writes/updates code (uses requirements from vector memory).
- Reviewer scores PEP8-style and returns JSON.
- Recommender proposes actionable fixes.
- Doc agent summarizes after each main step.
- All share a hub (for future messaging) and rate limits; logger attached.

In [ ]:
def coder_step(ctx: Context) -> AgentStep:
    reqs = vec_mem.search('calculator')
    requirements = reqs[0][0] if reqs else 'Build calculator'
    prompt = f"Build/Improve calculator code. Requirements: {requirements}. Current code:\n{ctx.scratch.get('code','')}"
    code = ctx.llm.generate(prompt)
    return AgentStep(out_messages=[Message(role='coder', content=code)], state_updates={'code': code})

def reviewer_step(ctx: Context) -> AgentStep:
    base = use_prompt('review_prompt')
    prompt = base + f"\nCode:\n{ctx.scratch.get('code','')}"
    review = ctx.llm.generate(prompt)
    try:
        import json
        parsed = json.loads(review)
        score = float(parsed.get('score', 0))
    except Exception:
        score = 0.0
    return AgentStep(out_messages=[Message(role='reviewer', content=review)], state_updates={'score': score, 'review': review})

def recommender_step(ctx: Context) -> AgentStep:
    prompt = f"Recommend specific fixes to reach 10/10. Review: {ctx.scratch.get('review','')}"
    recs = ctx.llm.generate(prompt)
    return AgentStep(out_messages=[Message(role='recommender', content=recs)], state_updates={'recs': recs})

def doc_step(ctx: Context) -> AgentStep:
    base = use_prompt('doc_prompt')
    prompt = base + f"\nCode excerpt:\n{ctx.scratch.get('code','')[:400]}\nReview: {ctx.scratch.get('review','')}\nRecs: {ctx.scratch.get('recs','')}"
    doc = ctx.llm.generate(prompt)
    return AgentStep(out_messages=[Message(role='doc', content=doc)], state_updates={'docs': doc})

coder = AgentBuilder(name='coder', role='dev').with_llm(llm).with_step(coder_step).build()
reviewer = AgentBuilder(name='reviewer', role='qa').with_llm(llm).with_step(reviewer_step).build()
recommender = AgentBuilder(name='recommender', role='advisor').with_llm(llm).with_step(recommender_step).build()
doc_agent = AgentBuilder(name='doc', role='documentation').with_llm(llm).with_step(doc_step).build()


## Orchestration: with_steps + GraphRunner
- with_steps loop: coder → reviewer → recommender with doc_agent after each.
- GraphRunner wraps the with_steps-driven agent for future DAG extension (e.g., add security reviewer dynamically).

In [ ]:
def loop_once(ctx: Context) -> AgentStep:
    steps = [
        StepSpec(name='code', fn=coder.run),
        StepSpec(name='review', fn=reviewer.run),
        StepSpec(name='recs', fn=recommender.run),
    ]
    # Run steps; after each main step, run doc agent
    for spec in steps:
        res = spec.fn(ctx)
        ctx.scratch.update(res.state_updates)
        doc_res = doc_agent.run(ctx)
        ctx.scratch.update(doc_res.state_updates)
    return AgentStep(out_messages=[], state_updates={})

loop_agent = AgentBuilder(name='loop', role='orchestrator').with_llm(llm).with_step(loop_once).build()

graph = {
    'loop': GraphNodeSpec(agent=loop_agent),
}
runner = GraphRunner(graph)

ctx = Context(goal='Ship calculator release', scratch={'score': 0.0}, components={'logger': logger})
result = runner.run(goal=ctx.goal, inputs=ctx.scratch)
ctx.scratch['score'] = ctx.scratch.get('score', 0.0)
summary = {
    'score': ctx.scratch.get('score', 0.0),
    'code_excerpt': ctx.scratch.get('code','')[:300],
    'docs_excerpt': ctx.scratch.get('docs','')[:300],
}
summary


## Parallel, async, batch, branching, comms, and logging
Below we show with_steps in parallel/batch mode (including async steps), then namespace communication with private and broadcast messaging, and simple logging.

In [ ]:
import asyncio
from agentic_codex import Context, StepSpec, run_steps, StructuredLogger
from agentic_codex.core.schemas import AgentStep, Message

async def async_worker(ctx: Context) -> AgentStep:
    await asyncio.sleep(0)
    item = ctx.scratch.get('batch_item')
    content = f'async_processed:{item}'
    return AgentStep(out_messages=[Message(role='async_worker', content=content)], state_updates={'processed': item})

logger_demo = StructuredLogger(name='demo')
ctx_parallel = Context(goal='parallel-async-demo', scratch={'items': [1,2,3]}, components={'logger': logger_demo})
specs = [StepSpec(name='async_batch', fn=async_worker, batch_key='items', parallel=True)]
parallel_results = run_steps(specs, ctx_parallel, max_parallel=3)
parallel_messages = [m.content for r in parallel_results for m in r.out_messages]
parallel_messages


In [ ]:
from agentic_codex import CommunicationHub, CommEnvelope

hub = CommunicationHub()
hub.create_namespace('team', ['coder','reviewer','recommender'])
hub.create_namespace('docs', ['doc'])
hub.link_namespaces('team', 'docs')

# Private doc updates after each iteration
hub.send(CommEnvelope(sender='doc', mode='namespace', targets=['docs'], content='iteration 1 summary', meta={'namespace':'docs'}))
# Broadcast when goal reached
hub.send(CommEnvelope(sender='loop', mode='broadcast', targets=[], content='Goal reached: score >= 9.5'))

[(rec.agent, rec.channel, rec.content) for rec in hub.conversation()]


## Memory and storage: SQLiteMemory + VectorMemory
Demonstrates lightweight key/value storage and vector retrieval (hash embeddings).

In [ ]:
from agentic_codex import SQLiteMemory, HashEmbeddingAdapter, VectorMemory
sql_mem = SQLiteMemory(':memory:')
sql_mem.put('cfg', 'release=true')
sql_search = sql_mem.search('release')
vec = VectorMemory(HashEmbeddingAdapter(dimension=16).embed)
vec.add('note', 'Calculator must support cli and logging')
vec_hits = vec.search('cli logging', k=1)
sql_search, vec_hits


## Observability: tracer + run store + Prometheus text
Trace spans/metrics and persist runs.

In [ ]:
from agentic_codex import StructuredLogger
from agentic_codex.core.observability.tracer import Tracer
from agentic_codex.core.observability.prometheus import render_metrics
from agentic_codex.core.schemas import RunResult, Message
from datetime import datetime
logger = StructuredLogger(name='obs-demo')
tracer = Tracer(run_id='demo', exporter=None)
with tracer.span('demo-span', step='example'):
    tracer.metric('demo.metric', 1, tag='x')
logger.log('demo.log', detail='span complete')
metrics = {'counters': {('demo.metric', (('tag','x'),)): 1}, 'latency': {}}
prom_text = render_metrics(metrics)
prom_text


## Prompt frameworks: switchable templates
Prompt_Framework generates prompts across frameworks (CoStar, CARE, etc.).

In [ ]:
from agentic_codex import Prompt_Framework
pf = Prompt_Framework(context='Refactor calculator for reliability', output_type='code review', style='formal')
pf.switch_framework('care')
care_prompt = pf.generate_prompt(action='refactor', result='clean code', example='use functions')
pf.switch_framework('costar')
costar_prompt = pf.generate_prompt(reasoning='list steps then write code')
care_prompt[:200], costar_prompt[:200]


## Dynamic graph mutation
GraphRunner can add nodes at runtime via `_graph_additions` in state updates.

In [ ]:
from agentic_codex import AgentBuilder, GraphRunner, GraphNodeSpec
from agentic_codex.core.schemas import AgentStep, Message
def base_step(ctx):
    # Add a new node 'extra' after first run
    ctx.scratch.setdefault('_graph_additions', []).append(('extra', GraphNodeSpec(agent=extra_agent)))
    return AgentStep(out_messages=[Message(role='base', content='base run')])
def extra_step(ctx):
    return AgentStep(out_messages=[Message(role='extra', content='extra run')])
base_agent = AgentBuilder(name='base', role='base').with_llm(llm).with_step(base_step).build()
extra_agent = AgentBuilder(name='extra', role='extra').with_llm(llm).with_step(extra_step).build()
graph = {'base': GraphNodeSpec(agent=base_agent)}
runner = GraphRunner(graph, branch_budget=1)
ctx_dyn = Context(goal='dyn graph', scratch={})
dyn_result = runner.run(goal=ctx_dyn.goal, inputs=ctx_dyn.scratch)
[(m.role, m.content) for m in dyn_result.messages]


## LLM JSON extraction with tags + regex
Enforce JSON inside `<json>...</json>` and parse defensively with regex before validation.

In [ ]:
import re, json
from agentic_codex import parse_llm_json

llm_output = '<json>{\"score\": 9.6, \"issues\": [\"spacing\", \"docstrings\"]}</json>'\n
match = re.search(r'<json>(.*?)</json>', llm_output, re.DOTALL)
if not match:
    raise ValueError('Missing <json> block')
json_block = match.group(1)
parsed = parse_llm_json(json_block)
parsed


## Comprehensive Feature Index (expanded)
This section catalogs the framework's core features, explains why each matters, shows where the implementation lives in the codebase, and describes concrete places you would extend or integrate the capability in a production project.

**Guidance:** For each feature below you'll find: a short description, the primary implementation files you can inspect or modify, example notebook cells that exercise the feature, and notes on extension points and safety considerations.

- **Skills & SkillRegistry**:
  - What: Small, function-like utilities (search, math, file ops) that can be registered and looked up by agents at runtime to keep heavy logic outside LLM prompts.
  - Implemented in: `agentic_codex/core/skills.py` (see `SkillRegistry`, `BUILTIN_SKILLS`).
  - Notebook demo: 'Skills + ToolCapability' cell and the `enhanced_step` which calls `ctx.registry.get(...)`.
  - Where to implement/integrate: register domain-specific skills (e.g., CI status fetchers, package registry clients) and attach via `SkillRegistryCapability` in `AgentBuilder` or manifests.
  - Notes: Keep skills deterministic and side-effect controlled; prefer idempotent operations or wrap stateful interactions with guards.

- **ToolCapability & Tool Adapters** (HTTP, RAG, DB, FS, Exec):
  - What: Adapters expose external actions (HTTP, DB queries, file I/O, sanitized code execution). Tools are wrapped with guards (permissions, budgets, rate limits) before being added to `Context`.
  - Implemented in: `agentic_codex/core/tools.py` (adapters and `guarded_tool`), wired into `Capability` in `agentic_codex/core/capabilities.py`.
  - Notebook demo: RAG demo (`RAGToolAdapter`), Budget enforcement demo, Mock MCP tool demo.
  - Where to implement/integrate: add new adapters for your infra (e.g., `S3ToolAdapter` or `PostgresToolAdapter`) in `core/tools.py` or a new `integrations/` package and surface them via `ToolCapability` in agent manifests or `AgentBuilder`.
  - Notes: Always wrap networked/privileged tools in `ToolPermissions` and `BudgetGuard` to enforce least privilege and limit costs. Use `MultiLimiter` for shared rate limits.

- **Permissions & BudgetGuard**:
  - What: Enforce which agents can call which tools and limit total usage to avoid runaway costs or damage.
  - Implemented in: `agentic_codex/core/tools.py` (`ToolPermissions`, `BudgetGuard`) and applied by `ToolCapability.bind`.
  - Notebook demo: Budget enforcement cell shows a budget exceed error captured in agent output.
  - Where to implement/integrate: define per-environment policies in manifests (see `manifests/validators.py`) and create environment-specific budgets in deployment code.

- **MessageBus / CommunicationHub**:
  - What: Publish/subscribe and namespace-based messaging between agents and components. Useful for loose coupling and event-driven workflows.
  - Implemented in: `agentic_codex/core/message_bus.py` and `agentic_codex/CommunicationHub` wrappers referenced in examples.
  - Notebook demo: `CommunicationHub` examples showing namespace sends and broadcasts.
  - Where to implement/integrate: use `MessageBusCapability` to attach a shared bus to agents created by `AgentBuilder`, or replace the bus with a remote-backed implementation for cross-process communication.

- **RunStore persistence & replay**:
  - What: Persist runs (messages, events, artifacts) as JSON for audit, replay, or debugging.
  - Implemented in: `agentic_codex/core/observability/run_store.py` (`RunStore`).
  - Notebook demo: `RunStore` usage in the enhanced demo and the RunStore replay cell showing how to list and load saved runs.
  - Where to implement/integrate: wire a `RunStore` instance into production services (e.g., `service/http.py` already references `RunStore`) and use CLI `agentic replay` to inspect runs.
  - Notes: Redact secrets before saving or use a secure store/encryption for sensitive runs.

- **Hot-reload manifests & loaders**:
  - What: Manifest files describe agents, tools, and workflows. Hot-reload watches manifest files and rebuilds runtime graphs without a full restart.
  - Implemented in: `agentic_codex/manifests/hot_reload.py`, `agentic_codex/manifests/loaders.py`, and validated by `manifests/validators.py`.
  - Notebook demo: Hot-reload demo that creates a temp manifest and triggers `watch_manifest`.
  - Where to implement/integrate: store manifests in a `manifests/` directory in production, and run the watcher in a lightweight supervisor to enable live config changes.

- **Cancellation & Cooperative Shutdown**:
  - What: `CancelToken` allows external controllers to ask agents to stop early (cooperative cancellation).
  - Implemented in: `agentic_codex/core/cancel.py` and checked by `Agent.run` and kernels.
  - Notebook demo: Cancellation token demo that cancels a long-running step from another thread.
  - Where to implement/integrate: pass a `CancelToken` in `Context.components['cancel_token']` from orchestration code (HTTP endpoints, CLI, or supervisors).

- **GraphRunner & Dynamic Graph Mutation**:
  - What: Execute DAGs of agent nodes, with runtime additions via `_graph_additions` in scratch/state for dynamic branching.
  - Implemented in: `agentic_codex/core/orchestration` (see `GraphRunner`, `GraphNodeSpec` references in examples).
  - Notebook demo: Dynamic graph mutation and conditional branching demos (security reviewer cell).
  - Where to implement/integrate: use `GraphRunner` for pipeline-style workflows (CI pipelines, multi-review flows) and implement guards/transforms in nodes.

- **Memory (scratch, episodic, vector, SQLite)**:
  - What: Multiple memory types for short-lived scratch data, episodic histories, and semantic vector retrieval.
  - Implemented in: `agentic_codex/core/memory.py` and `agentic_codex` top-level adapters (`SQLiteMemory`, `VectorMemory`, `HashEmbeddingAdapter`).
  - Notebook demo: VectorMemory and SQLiteMemory deep-dive cells.
  - Where to implement/integrate: pick `MemoryCapability` for agents that need persistent context; use vector memory for retrieval-augmented prompts and SQLiteMemory for small durable config/state.

- **Observability: Tracer, StructuredLogger, Prometheus metrics**:
  - What: Spans/metrics for each agent/node, structured logs (JSON), and prometheus-formatted text for scraping.


















**Next:** run the demo cells below to see each feature in action. For production hardening, follow the 'Where to implement' guidance to centralize guards, secrets, and rate-limiting in deployment code rather than notebooks.  - Where to implement/integrate: implement trainers that conform to `ReinforcementLearner` and attach via `AgentBuilder.with_capability(...)`; send signals (rewards) from `ToolAdapters` or `Kernel` learn hooks.  - Notebook demo: Minimal RL trainer stub that appends updates to `ctx.components['rl_history']`.  - Implemented in: `agentic_codex/core/capabilities.py` (see `ReinforcementLearningCapability`, `JEPALearningCapability`).  - What: Hook trainer logic into agent learn loops for RL-style feedback or joint-embedding predictive updates (JEPA).- **Reinforcement & JEPA hooks**:  - Notes: Treat these as defense-in-depth — combine with process-level sandboxing (containers) for stronger guarantees.  - Where to implement/integrate: use in any agent that must run user-provided code or read arbitrary files (e.g., plugin runners, sandboxed graders).  - Notebook demo: Safe exec and safe reader wiring earlier in the notebook; use them in `AgentStep` implementations to execute or read files safely.  - Implemented in: `agentic_codex/core/tools.py` (`SanitizedCodeExecutionToolAdapter`, `SafeFileReaderToolAdapter`).  - What: Tools that limit unsafe operations (no imports, restricted builtins, path restrictions).- **Safety: Sanitized execution and SafeFileReader**:  - Where to implement/integrate: attach tracer + logger to `Context.components` at orchestration start (service/http.py shows an example). Export spans to OTEL/Jaeger in production.  - Notebook demo: `tracer` and `render_metrics` usage in Observability cell; `StructuredLogger` usage in RunStore demo.  - Implemented in: `agentic_codex/core/observability/` (tracer.py, logger.py, prometheus.py, run_store.py).                

In [ ]:
# Demo: Skills + ToolCapability + Budgets + Reinforcement hook integrated into the flow
from agentic_codex.core.skills import SkillRegistry, BUILTIN_SKILLS
from agentic_codex.core.tools import MathToolAdapter, HTTPToolAdapter, ToolPermissions, BudgetGuard
from agentic_codex.core.capabilities import SkillRegistryCapability, ToolCapability, ReinforcementLearningCapability, LLMCapability
from agentic_codex import AgentBuilder, Context, RunStore, FunctionAdapter
from agentic_codex.core.cancel import CancelToken
import tempfile, json

# Skill registry with a small builtin skill
skills = SkillRegistry()
skills.register('math_solver', BUILTIN_SKILLS['math_solver'])

# Tools: math (fast local), http (demo). Name strings come from the adapters' .name
math_tool = MathToolAdapter()
http_tool = HTTPToolAdapter(timeout=1.0)

# Permissions: only the agent named 'enhanced_coder' can call math.eval
permissions = ToolPermissions(allowed_skills={'enhanced_coder': {math_tool.name}})
# Budget: allow 2 calls before raising
budget = BudgetGuard(max_calls=2)

# A tiny reinforcement trainer stub that records updates into context.components['rl_history']
def rl_trainer(ctx, step):
    hist = ctx.components.setdefault('rl_history', [])
    hist.append({'agent': getattr(step, 'agent', 'enhanced'), 'meta': getattr(step, 'state_updates', {})})

# Wrap the FunctionAdapter as a stub LLM (keeps notebook offline-friendly)
stub_llm = FunctionAdapter(lambda p: '[stub] ' + (p.splitlines()[-1] if p else 'ok'))

# Build an enhanced agent that uses skills + tools + RL hooks
def enhanced_step(ctx: Context):
    # Use a skill (math_solver) and a guarded tool (math.eval). Both are actively used.
    solver = ctx.registry.get('math_solver')
    solver_res = solver('1+2*3')
    tool = ctx.get_tool(math_tool.name)
    tool_res = tool.invoke(expression='1+2*3')
    # Push outputs as messages and state updates so RL hook and RunStore can capture them
    from agentic_codex.core.schemas import AgentStep, Message
    msg = Message(role='enhanced', content=f'skill={solver_res} tool={tool_res}')
    return AgentStep(out_messages=[msg], state_updates={'skill': solver_res, 'tool': tool_res})

enhanced = AgentBuilder(name='enhanced_coder', role='dev').with_capability(SkillRegistryCapability(skills)).with_capability(ToolCapability({'math.eval': math_tool, 'http.get': http_tool}, permissions=permissions, budget=budget)).with_capability(ReinforcementLearningCapability(trainer=rl_trainer, auto_setup=False)).with_llm(stub_llm).with_step(enhanced_step).build()

# Run: create context with RunStore and logger attached
store = RunStore('runs/')
ctx = Context(goal='enhanced-demo', scratch={'run_id': 'enhanced-demo-1'}, components={'logger': None})
res = enhanced.run(ctx)
# Persist a minimal run record so you can inspect later
store.save({'run_id': ctx.scratch.get('run_id', 'enhanced-demo-1'), 'messages': [m.model_dump() if hasattr(m, 'model_dump') else m.__dict__ for m in res.out_messages], 'meta': ctx.scratch})
res.out_messages, ctx.components.get('rl_history'), ctx.scratch['tool']


In [ ]:
# Demo: Budget enforcement - repeated tool calls will trigger BudgetGuard
from agentic_codex.core.tools import MathToolAdapter, BudgetGuard
from agentic_codex.core.capabilities import ToolCapability
from agentic_codex import AgentBuilder, Context, FunctionAdapter

math_tool = MathToolAdapter()
budget = BudgetGuard(max_calls=2)
stub_llm = FunctionAdapter(lambda p: 'ok')

def call_tool_step(ctx: Context):
    # Call the same tool three times to force a budget error on the 3rd call
    t = ctx.get_tool(math_tool.name)
    out = []
    for i in range(3):
        try:
            r = t.invoke(expression=f'{i}+1')
            out.append(r)
        except Exception as e:
            out.append({'error': str(e)})
    from agentic_codex.core.schemas import AgentStep, Message
    return AgentStep(out_messages=[Message(role='tester', content=str(out))], state_updates={})

agent = AgentBuilder(name='budget_tester', role='tester').with_capability(ToolCapability({'math.eval': math_tool}, budget=budget)).with_llm(stub_llm).with_step(call_tool_step).build()
c = Context(goal='budget-test')
r = agent.run(c)
r.out_messages[0].content


In [ ]:
# Demo: Hot-reload manifest (short-lived watch) - creates a temp manifest, watches it, and triggers on_change
from agentic_codex.manifests.hot_reload import watch_manifest
from agentic_codex.manifests.loaders import build_from_manifest
import threading, time, tempfile, pathlib

# Create a small manifest file (minimal key:value style parsable by loader)
tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.yaml')
path = pathlib.Path(tmp.name)
tmp.write(b'
# demo manifest
name: demo_workflow
steps: []
')
tmp.close()

change_events = []
def on_change(manifest):
    change_events.append(manifest)

stop = threading.Event()
thr = watch_manifest(path, on_change, interval=0.2, stop_event=stop)
# Update the file to trigger the watcher once
time.sleep(0.25)
path.write_text('# updated
name: demo_workflow
steps: []
')
time.sleep(0.4)
stop.set()
time.sleep(0.1)
len(change_events)


In [ ]:
# Demo: Cancellation token - agent observes token and exits early
from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.cancel import CancelToken
from agentic_codex.core.schemas import AgentStep, Message

# A long-running step that checks for cancellation via context.components['cancel_token']
def long_step(ctx: Context):
    import time
    token = ctx.components.get('cancel_token')
    for i in range(5):
        if token and getattr(token, 'cancelled', False):
            return AgentStep(out_messages=[Message(role='long', content='cancelled')], state_updates={}, stop=True)
        time.sleep(0.05)
    return AgentStep(out_messages=[Message(role='long', content='done')], state_updates={})

agent = AgentBuilder(name='long_runner', role='worker').with_llm(FunctionAdapter(lambda p: 'ok')).with_step(long_step).build()
token = CancelToken()
ctx = Context(goal='cancel-demo', components={'cancel_token': token})
# Cancel after a short delay from another thread
import threading
def canceller():
    import time; time.sleep(0.08); token.cancel('stop')
threading.Thread(target=canceller).start()
res = agent.run(ctx)
res.out_messages[0].content


## Additional Features & Deep Dives
Below are short deep-dive notes added to the previous index that describe practical implementation and integration advice for each demo block added in this notebook.

- **RAG / Retrieval-Augmented Generation**:
  - Where it's implemented in the repo: `agentic_codex/core/tools.py` (`RAGToolAdapter`).
  - How it is used here: the RAG tool is registered and then called from an agent step to fetch supporting documents that are inserted into prompts or returned directly.
  - Production tips: replace the in-memory index with a real vector DB or search service; ensure retrieved documents are sanitized and provenance is stored (use `RunStore` to record which docs were used).

- **VectorMemory deep-dive**:
  - Where it's implemented: `agentic_codex` top-level adapters (e.g., `VectorMemory`, `HashEmbeddingAdapter`).
  - How it is used here: we add several documents, compute embeddings via the hash embedder, and run nearest-neighbor searches to support agent decisions.
  - Production tips: swap `HashEmbeddingAdapter` for a real embedder (OpenAI/embedding model); persist embeddings to a vector DB; version embeddings; precompute for large corpora.

- **SQLiteMemory persistence**:
  - Where it's implemented: `agentic_codex` memory adapters (see `SQLiteMemory` in top-level exports).
  - How it is used here: store small config and state entries to survive process restarts.
  - Production tips: use a managed SQL store for cross-process durability; apply migrations for schema changes and avoid storing secrets in plaintext.

- **RunStore replay & StructuredLogger**:
  - Where it's implemented: `agentic_codex/core/observability/run_store.py` and `agentic_codex/core/observability/logger.py`.
  - How it is used here: the enhanced demo saves a minimal run JSON to `runs/` and the replay cell loads the run to show messages and metadata.
  - Production tips: integrate `RunStore` with an append-only backing store (object storage or DB), and redact PII before persisting. Provide a CLI `agentic replay` (existing CLI references) for operators.

- **Mock MCP tool registration**:
  - Where it's implemented: `agentic_codex/core/tools.py` plus manifests support (`agentic_codex/manifests/*`) for describing remote MCP endpoints and tools.
  - How it is used here: we register a `MockMCPAdapter` and call it from an agent; in production, MCP adapters would call remote servers and expose metadata.
  - Production tips: secure MCP connections with mTLS, add retry/backoff, and ensure rate limits and budgets are applied.

- **Conditional GraphRunner branching**:
  - Where it's implemented: `agentic_codex/core/orchestration` (GraphRunner and GraphNodeSpec usage are shown in examples).
  - How it is used here: base node adds a `security` node into `_graph_additions` based on a low score; GraphRunner runs additional nodes dynamically.



**Comment style & documentation:** each demo cell includes inline comments describing purpose, where to find the core implementation, and how to replace notebook stubs with real integrations (embedding models, DBs, remote MCP endpoints). Use the `examples/` folder as canonical runnable demonstrations when promoting these patterns into CI or docs.  - Production tips: attach guards/validators on `_graph_additions` to avoid unbounded graph growth, and set `branch_budget` to limit expansion.                

In [ ]:
# 1) RAG demo: local index + RAGToolAdapter usage
# Explanation: RAGToolAdapter mimics a lightweight retrieval system. Agents can call it as a tool
# to fetch context which can be injected into prompts for better factual grounding.
from agentic_codex.core.tools import RAGToolAdapter
from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.schemas import AgentStep, Message

# Build a tiny in-memory index mapping short keys to documents
index = {"calc_design": "Calculator must be accurate to 1e-6 and have CLI", "user_guide": "Usage: calc --add --sub --mul --div"}
rag = RAGToolAdapter(index=index)

# Agent that uses rag.query as a tool to fetch context and respond
def rag_step(ctx: Context):
    # Call the RAG tool (available under rag.name)
    tool = ctx.get_tool(rag.name)
    res = tool.invoke(query='calculator', k=2)
    # Create an AgentStep that returns the retrieved results as a message
    return AgentStep(out_messages=[Message(role='rag', content=str(res['results']))], state_updates={'rag_results': res['results']})

rag_agent = AgentBuilder(name='rag_user', role='user').with_capability( 
    # surface the RAG tool under the agent's tools capability
    __import__('agentic_codex').agentic_codex.core.capabilities.ToolCapability({'rag.query': rag})).with_llm(FunctionAdapter(lambda p: 'ok')).with_step(rag_step).build()

# Simpler: register tool directly into context for this quick demo
ctx = Context(goal='rag-demo')
ctx.add_tool(rag.name, rag)
out = rag_agent.run(ctx)
out.out_messages[0].content

# Agentic Components Feature Showcase

This notebook walks through the core features of the Agentic Components framework and ends with a complex, production-style use case. It is heavily annotated so you can adapt it quickly.

### What you'll see
- Lego-style agents (`AgentBuilder`) with capabilities (LLM, tools, memory, prompts, rate limits).
- Prompt management (Prompt-Framework optional) with versioning.
- with_steps (batch/parallel/conditional) and GraphRunner (DAG with dynamic additions).
- CommunicationHub (direct/broadcast/namespace) + rate limits and logging/metrics.
- Memory options (scratch, episodic, semantic, vector) with an in-notebook embedding + retrieval.
- Safe tools (sanitized code exec, safe file read).
- Run persistence (RunStore) and structured logging hooks.
- A complex "Release Readiness" use case tying everything together.

In [ ]:
# Runtime setup: optional OpenAI key with fallback stub LLM
import os
from getpass import getpass

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter OPENAI_API_KEY (press Enter to skip): ')

from agentic_codex import (
    AgentBuilder, Context, FunctionAdapter, EnvOpenAIAdapter, EnvOpenAIEmbeddingAdapter,
    CommunicationHub, CommEnvelope, PromptManager,
    MultiLimiter, RateLimiter, SafeFileReaderToolAdapter, SanitizedCodeExecutionToolAdapter,
    HashEmbeddingAdapter, VectorMemory, RunStore, StructuredLogger,
    StepSpec, run_steps, GraphRunner, GraphNodeSpec, apply_rate_limits, load_prompts
)
from agentic_codex.core.schemas import AgentStep, Message

def stub_llm(prompt: str) -> str:
    lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
    return '[stub llm] ' + (lines[-1][:200] if lines else 'response')

try:
    llm = EnvOpenAIAdapter(model='gpt-4o-mini')
except Exception:
    llm = FunctionAdapter(stub_llm)


## Prompt management (versioned prompts)
- Uses Prompt-Framework if installed (`pip install agentic-codex[prompt]`).
- Falls back to a simple dict if the optional dependency is missing.

In [ ]:
prompt_manager = PromptManager()
prompt_store = {}
try:
    prompt_manager.add('review_prompt', 'Review code and score 0-10; respond JSON.', version='v1')
    prompt_manager.add('doc_prompt', 'Document the changes and risks.', version='v1')
    use_prompt = lambda name: prompt_manager.get(name)
except Exception:
    prompt_store['review_prompt'] = 'Review code and score 0-10; respond JSON.'
    prompt_store['doc_prompt'] = 'Document the changes and risks.'
    use_prompt = lambda name: prompt_store.get(name, '')


## Memory, tools, comms, limits, logging
- VectorMemory with hash embeddings for local retrieval.
- Safe tools: sanitized code exec + safe file reader.
- Communication hub with namespace rate limits.
- RunStore for persistence (JSON); StructuredLogger for JSON logs.

In [ ]:
embedder = HashEmbeddingAdapter(dimension=64)
vec_mem = VectorMemory(embedder.embed)
vec_mem.add('calc_requirements', 'Calculator must support add/subtract/multiply/divide with CLI entry point')

safe_reader = SafeFileReaderToolAdapter(root='.')
safe_exec = SanitizedCodeExecutionToolAdapter()

hub = CommunicationHub()
hub.create_namespace('engineering', ['coder', 'reviewer', 'recommender', 'doc'])
hub.set_namespace_rate_limit('engineering', rate=5, capacity=5)

limiter = MultiLimiter()
limiter.set_limit('tool', safe_reader.name, RateLimiter(rate_per_sec=5, capacity=5))
limiter.set_limit('tool', safe_exec.name, RateLimiter(rate_per_sec=3, capacity=3))

store = RunStore('runs/')
logger = StructuredLogger(name='agentic-demo')


## Agents for complex release-readiness use case
- Coder writes/updates code (uses requirements from vector memory).
- Reviewer scores PEP8-style and returns JSON.
- Recommender proposes actionable fixes.
- Doc agent summarizes after each main step.
- All share a hub (for future messaging) and rate limits; logger attached.

In [ ]:
def coder_step(ctx: Context) -> AgentStep:
    reqs = vec_mem.search('calculator')
    requirements = reqs[0][0] if reqs else 'Build calculator'
    prompt = f"Build/Improve calculator code. Requirements: {requirements}. Current code:\n{ctx.scratch.get('code','')}"
    code = ctx.llm.generate(prompt)
    return AgentStep(out_messages=[Message(role='coder', content=code)], state_updates={'code': code})

def reviewer_step(ctx: Context) -> AgentStep:
    base = use_prompt('review_prompt')
    prompt = base + f"\nCode:\n{ctx.scratch.get('code','')}"
    review = ctx.llm.generate(prompt)
    try:
        import json
        parsed = json.loads(review)
        score = float(parsed.get('score', 0))
    except Exception:
        score = 0.0
    return AgentStep(out_messages=[Message(role='reviewer', content=review)], state_updates={'score': score, 'review': review})

def recommender_step(ctx: Context) -> AgentStep:
    prompt = f"Recommend specific fixes to reach 10/10. Review: {ctx.scratch.get('review','')}"
    recs = ctx.llm.generate(prompt)
    return AgentStep(out_messages=[Message(role='recommender', content=recs)], state_updates={'recs': recs})

def doc_step(ctx: Context) -> AgentStep:
    base = use_prompt('doc_prompt')
    prompt = base + f"\nCode excerpt:\n{ctx.scratch.get('code','')[:400]}\nReview: {ctx.scratch.get('review','')}\nRecs: {ctx.scratch.get('recs','')}"
    doc = ctx.llm.generate(prompt)
    return AgentStep(out_messages=[Message(role='doc', content=doc)], state_updates={'docs': doc})

coder = AgentBuilder(name='coder', role='dev').with_llm(llm).with_step(coder_step).build()
reviewer = AgentBuilder(name='reviewer', role='qa').with_llm(llm).with_step(reviewer_step).build()
recommender = AgentBuilder(name='recommender', role='advisor').with_llm(llm).with_step(recommender_step).build()
doc_agent = AgentBuilder(name='doc', role='documentation').with_llm(llm).with_step(doc_step).build()


## Orchestration: with_steps + GraphRunner
- with_steps loop: coder → reviewer → recommender with doc_agent after each.
- GraphRunner wraps the with_steps-driven agent for future DAG extension (e.g., add security reviewer dynamically).

In [ ]:
def loop_once(ctx: Context) -> AgentStep:
    steps = [
        StepSpec(name='code', fn=coder.run),
        StepSpec(name='review', fn=reviewer.run),
        StepSpec(name='recs', fn=recommender.run),
    ]
    # Run steps; after each main step, run doc agent
    for spec in steps:
        res = spec.fn(ctx)
        ctx.scratch.update(res.state_updates)
        doc_res = doc_agent.run(ctx)
        ctx.scratch.update(doc_res.state_updates)
    return AgentStep(out_messages=[], state_updates={})

loop_agent = AgentBuilder(name='loop', role='orchestrator').with_llm(llm).with_step(loop_once).build()

graph = {
    'loop': GraphNodeSpec(agent=loop_agent),
}
runner = GraphRunner(graph)

ctx = Context(goal='Ship calculator release', scratch={'score': 0.0}, components={'logger': logger})
result = runner.run(goal=ctx.goal, inputs=ctx.scratch)
ctx.scratch['score'] = ctx.scratch.get('score', 0.0)
summary = {
    'score': ctx.scratch.get('score', 0.0),
    'code_excerpt': ctx.scratch.get('code','')[:300],
    'docs_excerpt': ctx.scratch.get('docs','')[:300],
}
summary


## Parallel, async, batch, branching, comms, and logging
Below we show with_steps in parallel/batch mode (including async steps), then namespace communication with private and broadcast messaging, and simple logging.

In [ ]:
import asyncio
from agentic_codex import Context, StepSpec, run_steps, StructuredLogger
from agentic_codex.core.schemas import AgentStep, Message

async def async_worker(ctx: Context) -> AgentStep:
    await asyncio.sleep(0)
    item = ctx.scratch.get('batch_item')
    content = f'async_processed:{item}'
    return AgentStep(out_messages=[Message(role='async_worker', content=content)], state_updates={'processed': item})

logger_demo = StructuredLogger(name='demo')
ctx_parallel = Context(goal='parallel-async-demo', scratch={'items': [1,2,3]}, components={'logger': logger_demo})
specs = [StepSpec(name='async_batch', fn=async_worker, batch_key='items', parallel=True)]
parallel_results = run_steps(specs, ctx_parallel, max_parallel=3)
parallel_messages = [m.content for r in parallel_results for m in r.out_messages]
parallel_messages


In [ ]:
from agentic_codex import CommunicationHub, CommEnvelope

hub = CommunicationHub()
hub.create_namespace('team', ['coder','reviewer','recommender'])
hub.create_namespace('docs', ['doc'])
hub.link_namespaces('team', 'docs')

# Private doc updates after each iteration
hub.send(CommEnvelope(sender='doc', mode='namespace', targets=['docs'], content='iteration 1 summary', meta={'namespace':'docs'}))
# Broadcast when goal reached
hub.send(CommEnvelope(sender='loop', mode='broadcast', targets=[], content='Goal reached: score >= 9.5'))

[(rec.agent, rec.channel, rec.content) for rec in hub.conversation()]


## Memory and storage: SQLiteMemory + VectorMemory
Demonstrates lightweight key/value storage and vector retrieval (hash embeddings).

In [ ]:
from agentic_codex import SQLiteMemory, HashEmbeddingAdapter, VectorMemory
sql_mem = SQLiteMemory(':memory:')
sql_mem.put('cfg', 'release=true')
sql_search = sql_mem.search('release')
vec = VectorMemory(HashEmbeddingAdapter(dimension=16).embed)
vec.add('note', 'Calculator must support cli and logging')
vec_hits = vec.search('cli logging', k=1)
sql_search, vec_hits


## Observability: tracer + run store + Prometheus text
Trace spans/metrics and persist runs.

In [ ]:
from agentic_codex import StructuredLogger
from agentic_codex.core.observability.tracer import Tracer
from agentic_codex.core.observability.prometheus import render_metrics
from agentic_codex.core.schemas import RunResult, Message
from datetime import datetime
logger = StructuredLogger(name='obs-demo')
tracer = Tracer(run_id='demo', exporter=None)
with tracer.span('demo-span', step='example'):
    tracer.metric('demo.metric', 1, tag='x')
logger.log('demo.log', detail='span complete')
metrics = {'counters': {('demo.metric', (('tag','x'),)): 1}, 'latency': {}}
prom_text = render_metrics(metrics)
prom_text


## Prompt frameworks: switchable templates
Prompt_Framework generates prompts across frameworks (CoStar, CARE, etc.).

In [ ]:
from agentic_codex import Prompt_Framework
pf = Prompt_Framework(context='Refactor calculator for reliability', output_type='code review', style='formal')
pf.switch_framework('care')
care_prompt = pf.generate_prompt(action='refactor', result='clean code', example='use functions')
pf.switch_framework('costar')
costar_prompt = pf.generate_prompt(reasoning='list steps then write code')
care_prompt[:200], costar_prompt[:200]


## Dynamic graph mutation
GraphRunner can add nodes at runtime via `_graph_additions` in state updates.

In [ ]:
from agentic_codex import AgentBuilder, GraphRunner, GraphNodeSpec
from agentic_codex.core.schemas import AgentStep, Message
def base_step(ctx):
    # Add a new node 'extra' after first run
    ctx.scratch.setdefault('_graph_additions', []).append(('extra', GraphNodeSpec(agent=extra_agent)))
    return AgentStep(out_messages=[Message(role='base', content='base run')])
def extra_step(ctx):
    return AgentStep(out_messages=[Message(role='extra', content='extra run')])
base_agent = AgentBuilder(name='base', role='base').with_llm(llm).with_step(base_step).build()
extra_agent = AgentBuilder(name='extra', role='extra').with_llm(llm).with_step(extra_step).build()
graph = {'base': GraphNodeSpec(agent=base_agent)}
runner = GraphRunner(graph, branch_budget=1)
ctx_dyn = Context(goal='dyn graph', scratch={})
dyn_result = runner.run(goal=ctx_dyn.goal, inputs=ctx_dyn.scratch)
[(m.role, m.content) for m in dyn_result.messages]


## LLM JSON extraction with tags + regex
Enforce JSON inside `<json>...</json>` and parse defensively with regex before validation.

In [ ]:
import re, json
from agentic_codex import parse_llm_json

llm_output = '<json>{\"score\": 9.6, \"issues\": [\"spacing\", \"docstrings\"]}</json>'\n
match = re.search(r'<json>(.*?)</json>', llm_output, re.DOTALL)
if not match:
    raise ValueError('Missing <json> block')
json_block = match.group(1)
parsed = parse_llm_json(json_block)
parsed


## Comprehensive Feature Index (expanded)
This section catalogs the framework's core features, explains why each matters, shows where the implementation lives in the codebase, and describes concrete places you would extend or integrate the capability in a production project.

**Guidance:** For each feature below you'll find: a short description, the primary implementation files you can inspect or modify, example notebook cells that exercise the feature, and notes on extension points and safety considerations.

- **Skills & SkillRegistry**:
  - What: Small, function-like utilities (search, math, file ops) that can be registered and looked up by agents at runtime to keep heavy logic outside LLM prompts.
  - Implemented in: `agentic_codex/core/skills.py` (see `SkillRegistry`, `BUILTIN_SKILLS`).
  - Notebook demo: 'Skills + ToolCapability' cell and the `enhanced_step` which calls `ctx.registry.get(...)`.
  - Where to implement/integrate: register domain-specific skills (e.g., CI status fetchers, package registry clients) and attach via `SkillRegistryCapability` in `AgentBuilder` or manifests.
  - Notes: Keep skills deterministic and side-effect controlled; prefer idempotent operations or wrap stateful interactions with guards.

- **ToolCapability & Tool Adapters** (HTTP, RAG, DB, FS, Exec):
  - What: Adapters expose external actions (HTTP, DB queries, file I/O, sanitized code execution). Tools are wrapped with guards (permissions, budgets, rate limits) before being added to `Context`.
  - Implemented in: `agentic_codex/core/tools.py` (adapters and `guarded_tool`), wired into `Capability` in `agentic_codex/core/capabilities.py`.
  - Notebook demo: RAG demo (`RAGToolAdapter`), Budget enforcement demo, Mock MCP tool demo.
  - Where to implement/integrate: add new adapters for your infra (e.g., `S3ToolAdapter` or `PostgresToolAdapter`) in `core/tools.py` or a new `integrations/` package and surface them via `ToolCapability` in agent manifests or `AgentBuilder`.
  - Notes: Always wrap networked/privileged tools in `ToolPermissions` and `BudgetGuard` to enforce least privilege and limit costs. Use `MultiLimiter` for shared rate limits.

- **Permissions & BudgetGuard**:
  - What: Enforce which agents can call which tools and limit total usage to avoid runaway costs or damage.
  - Implemented in: `agentic_codex/core/tools.py` (`ToolPermissions`, `BudgetGuard`) and applied by `ToolCapability.bind`.
  - Notebook demo: Budget enforcement cell shows a budget exceed error captured in agent output.
  - Where to implement/integrate: define per-environment policies in manifests (see `manifests/validators.py`) and create environment-specific budgets in deployment code.

- **MessageBus / CommunicationHub**:
  - What: Publish/subscribe and namespace-based messaging between agents and components. Useful for loose coupling and event-driven workflows.
  - Implemented in: `agentic_codex/core/message_bus.py` and `agentic_codex/CommunicationHub` wrappers referenced in examples.
  - Notebook demo: `CommunicationHub` examples showing namespace sends and broadcasts.
  - Where to implement/integrate: use `MessageBusCapability` to attach a shared bus to agents created by `AgentBuilder`, or replace the bus with a remote-backed implementation for cross-process communication.

- **RunStore persistence & replay**:
  - What: Persist runs (messages, events, artifacts) as JSON for audit, replay, or debugging.
  - Implemented in: `agentic_codex/core/observability/run_store.py` (`RunStore`).
  - Notebook demo: `RunStore` usage in the enhanced demo and the RunStore replay cell showing how to list and load saved runs.
  - Where to implement/integrate: wire a `RunStore` instance into production services (e.g., `service/http.py` already references `RunStore`) and use CLI `agentic replay` to inspect runs.
  - Notes: Redact secrets before saving or use a secure store/encryption for sensitive runs.

- **Hot-reload manifests & loaders**:
  - What: Manifest files describe agents, tools, and workflows. Hot-reload watches manifest files and rebuilds runtime graphs without a full restart.
  - Implemented in: `agentic_codex/manifests/hot_reload.py`, `agentic_codex/manifests/loaders.py`, and validated by `manifests/validators.py`.
  - Notebook demo: Hot-reload demo that creates a temp manifest and triggers `watch_manifest`.
  - Where to implement/integrate: store manifests in a `manifests/` directory in production, and run the watcher in a lightweight supervisor to enable live config changes.

- **Cancellation & Cooperative Shutdown**:
  - What: `CancelToken` allows external controllers to ask agents to stop early (cooperative cancellation).
  - Implemented in: `agentic_codex/core/cancel.py` and checked by `Agent.run` and kernels.
  - Notebook demo: Cancellation token demo that cancels a long-running step from another thread.
  - Where to implement/integrate: pass a `CancelToken` in `Context.components['cancel_token']` from orchestration code (HTTP endpoints, CLI, or supervisors).

- **GraphRunner & Dynamic Graph Mutation**:
  - What: Execute DAGs of agent nodes, with runtime additions via `_graph_additions` in scratch/state for dynamic branching.
  - Implemented in: `agentic_codex/core/orchestration` (see `GraphRunner`, `GraphNodeSpec` references in examples).
  - Notebook demo: Dynamic graph mutation and conditional branching demos (security reviewer cell).
  - Where to implement/integrate: use `GraphRunner` for pipeline-style workflows (CI pipelines, multi-review flows) and implement guards/transforms in nodes.

- **Memory (scratch, episodic, vector, SQLite)**:
  - What: Multiple memory types for short-lived scratch data, episodic histories, and semantic vector retrieval.
  - Implemented in: `agentic_codex/core/memory.py` and `agentic_codex` top-level adapters (`SQLiteMemory`, `VectorMemory`, `HashEmbeddingAdapter`).
  - Notebook demo: VectorMemory and SQLiteMemory deep-dive cells.
  - Where to implement/integrate: pick `MemoryCapability` for agents that need persistent context; use vector memory for retrieval-augmented prompts and SQLiteMemory for small durable config/state.

- **Observability: Tracer, StructuredLogger, Prometheus metrics**:
  - What: Spans/metrics for each agent/node, structured logs (JSON), and prometheus-formatted text for scraping.


















**Next:** run the demo cells below to see each feature in action. For production hardening, follow the 'Where to implement' guidance to centralize guards, secrets, and rate-limiting in deployment code rather than notebooks.  - Where to implement/integrate: implement trainers that conform to `ReinforcementLearner` and attach via `AgentBuilder.with_capability(...)`; send signals (rewards) from `ToolAdapters` or `Kernel` learn hooks.  - Notebook demo: Minimal RL trainer stub that appends updates to `ctx.components['rl_history']`.  - Implemented in: `agentic_codex/core/capabilities.py` (see `ReinforcementLearningCapability`, `JEPALearningCapability`).  - What: Hook trainer logic into agent learn loops for RL-style feedback or joint-embedding predictive updates (JEPA).- **Reinforcement & JEPA hooks**:  - Notes: Treat these as defense-in-depth — combine with process-level sandboxing (containers) for stronger guarantees.  - Where to implement/integrate: use in any agent that must run user-provided code or read arbitrary files (e.g., plugin runners, sandboxed graders).  - Notebook demo: Safe exec and safe reader wiring earlier in the notebook; use them in `AgentStep` implementations to execute or read files safely.  - Implemented in: `agentic_codex/core/tools.py` (`SanitizedCodeExecutionToolAdapter`, `SafeFileReaderToolAdapter`).  - What: Tools that limit unsafe operations (no imports, restricted builtins, path restrictions).- **Safety: Sanitized execution and SafeFileReader**:  - Where to implement/integrate: attach tracer + logger to `Context.components` at orchestration start (service/http.py shows an example). Export spans to OTEL/Jaeger in production.  - Notebook demo: `tracer` and `render_metrics` usage in Observability cell; `StructuredLogger` usage in RunStore demo.  - Implemented in: `agentic_codex/core/observability/` (tracer.py, logger.py, prometheus.py, run_store.py).                

In [ ]:
# Demo: Skills + ToolCapability + Budgets + Reinforcement hook integrated into the flow
from agentic_codex.core.skills import SkillRegistry, BUILTIN_SKILLS
from agentic_codex.core.tools import MathToolAdapter, HTTPToolAdapter, ToolPermissions, BudgetGuard
from agentic_codex.core.capabilities import SkillRegistryCapability, ToolCapability, ReinforcementLearningCapability, LLMCapability
from agentic_codex import AgentBuilder, Context, RunStore, FunctionAdapter
from agentic_codex.core.cancel import CancelToken
import tempfile, json

# Skill registry with a small builtin skill
skills = SkillRegistry()
skills.register('math_solver', BUILTIN_SKILLS['math_solver'])

# Tools: math (fast local), http (demo). Name strings come from the adapters' .name
math_tool = MathToolAdapter()
http_tool = HTTPToolAdapter(timeout=1.0)

# Permissions: only the agent named 'enhanced_coder' can call math.eval
permissions = ToolPermissions(allowed_skills={'enhanced_coder': {math_tool.name}})
# Budget: allow 2 calls before raising
budget = BudgetGuard(max_calls=2)

# A tiny reinforcement trainer stub that records updates into context.components['rl_history']
def rl_trainer(ctx, step):
    hist = ctx.components.setdefault('rl_history', [])
    hist.append({'agent': getattr(step, 'agent', 'enhanced'), 'meta': getattr(step, 'state_updates', {})})

# Wrap the FunctionAdapter as a stub LLM (keeps notebook offline-friendly)
stub_llm = FunctionAdapter(lambda p: '[stub] ' + (p.splitlines()[-1] if p else 'ok'))

# Build an enhanced agent that uses skills + tools + RL hooks
def enhanced_step(ctx: Context):
    # Use a skill (math_solver) and a guarded tool (math.eval). Both are actively used.
    solver = ctx.registry.get('math_solver')
    solver_res = solver('1+2*3')
    tool = ctx.get_tool(math_tool.name)
    tool_res = tool.invoke(expression='1+2*3')
    # Push outputs as messages and state updates so RL hook and RunStore can capture them
    from agentic_codex.core.schemas import AgentStep, Message
    msg = Message(role='enhanced', content=f'skill={solver_res} tool={tool_res}')
    return AgentStep(out_messages=[msg], state_updates={'skill': solver_res, 'tool': tool_res})

enhanced = AgentBuilder(name='enhanced_coder', role='dev').with_capability(SkillRegistryCapability(skills)).with_capability(ToolCapability({'math.eval': math_tool, 'http.get': http_tool}, permissions=permissions, budget=budget)).with_capability(ReinforcementLearningCapability(trainer=rl_trainer, auto_setup=False)).with_llm(stub_llm).with_step(enhanced_step).build()

# Run: create context with RunStore and logger attached
store = RunStore('runs/')
ctx = Context(goal='enhanced-demo', scratch={'run_id': 'enhanced-demo-1'}, components={'logger': None})
res = enhanced.run(ctx)
# Persist a minimal run record so you can inspect later
store.save({'run_id': ctx.scratch.get('run_id', 'enhanced-demo-1'), 'messages': [m.model_dump() if hasattr(m, 'model_dump') else m.__dict__ for m in res.out_messages], 'meta': ctx.scratch})
res.out_messages, ctx.components.get('rl_history'), ctx.scratch['tool']


In [ ]:
# Demo: Budget enforcement - repeated tool calls will trigger BudgetGuard
from agentic_codex.core.tools import MathToolAdapter, BudgetGuard
from agentic_codex.core.capabilities import ToolCapability
from agentic_codex import AgentBuilder, Context, FunctionAdapter

math_tool = MathToolAdapter()
budget = BudgetGuard(max_calls=2)
stub_llm = FunctionAdapter(lambda p: 'ok')

def call_tool_step(ctx: Context):
    # Call the same tool three times to force a budget error on the 3rd call
    t = ctx.get_tool(math_tool.name)
    out = []
    for i in range(3):
        try:
            r = t.invoke(expression=f'{i}+1')
            out.append(r)
        except Exception as e:
            out.append({'error': str(e)})
    from agentic_codex.core.schemas import AgentStep, Message
    return AgentStep(out_messages=[Message(role='tester', content=str(out))], state_updates={})

agent = AgentBuilder(name='budget_tester', role='tester').with_capability(ToolCapability({'math.eval': math_tool}, budget=budget)).with_llm(stub_llm).with_step(call_tool_step).build()
c = Context(goal='budget-test')
r = agent.run(c)
r.out_messages[0].content


In [ ]:
# Demo: Hot-reload manifest (short-lived watch) - creates a temp manifest, watches it, and triggers on_change
from agentic_codex.manifests.hot_reload import watch_manifest
from agentic_codex.manifests.loaders import build_from_manifest
import threading, time, tempfile, pathlib

# Create a small manifest file (minimal key:value style parsable by loader)
tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.yaml')
path = pathlib.Path(tmp.name)
tmp.write(b'
# demo manifest
name: demo_workflow
steps: []
')
tmp.close()

change_events = []
def on_change(manifest):
    change_events.append(manifest)

stop = threading.Event()
thr = watch_manifest(path, on_change, interval=0.2, stop_event=stop)
# Update the file to trigger the watcher once
time.sleep(0.25)
path.write_text('# updated
name: demo_workflow
steps: []
')
time.sleep(0.4)
stop.set()
time.sleep(0.1)
len(change_events)


In [ ]:
# Demo: Cancellation token - agent observes token and exits early
from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.cancel import CancelToken
from agentic_codex.core.schemas import AgentStep, Message

# A long-running step that checks for cancellation via context.components['cancel_token']
def long_step(ctx: Context):
    import time
    token = ctx.components.get('cancel_token')
    for i in range(5):
        if token and getattr(token, 'cancelled', False):
            return AgentStep(out_messages=[Message(role='long', content='cancelled')], state_updates={}, stop=True)
        time.sleep(0.05)
    return AgentStep(out_messages=[Message(role='long', content='done')], state_updates={})

agent = AgentBuilder(name='long_runner', role='worker').with_llm(FunctionAdapter(lambda p: 'ok')).with_step(long_step).build()
token = CancelToken()
ctx = Context(goal='cancel-demo', components={'cancel_token': token})
# Cancel after a short delay from another thread
import threading
def canceller():
    import time; time.sleep(0.08); token.cancel('stop')
threading.Thread(target=canceller).start()
res = agent.run(ctx)
res.out_messages[0].content


## Additional Features & Deep Dives
Below are short deep-dive notes added to the previous index that describe practical implementation and integration advice for each demo block added in this notebook.

- **RAG / Retrieval-Augmented Generation**:
  - Where it's implemented in the repo: `agentic_codex/core/tools.py` (`RAGToolAdapter`).
  - How it is used here: the RAG tool is registered and then called from an agent step to fetch supporting documents that are inserted into prompts or returned directly.
  - Production tips: replace the in-memory index with a real vector DB or search service; ensure retrieved documents are sanitized and provenance is stored (use `RunStore` to record which docs were used).

- **VectorMemory deep-dive**:
  - Where it's implemented: `agentic_codex` top-level adapters (e.g., `VectorMemory`, `HashEmbeddingAdapter`).
  - How it is used here: we add several documents, compute embeddings via the hash embedder, and run nearest-neighbor searches to support agent decisions.
  - Production tips: swap `HashEmbeddingAdapter` for a real embedder (OpenAI/embedding model); persist embeddings to a vector DB; version embeddings; precompute for large corpora.

- **SQLiteMemory persistence**:
  - Where it's implemented: `agentic_codex` memory adapters (see `SQLiteMemory` in top-level exports).
  - How it is used here: store small config and state entries to survive process restarts.
  - Production tips: use a managed SQL store for cross-process durability; apply migrations for schema changes and avoid storing secrets in plaintext.

- **RunStore replay & StructuredLogger**:
  - Where it's implemented: `agentic_codex/core/observability/run_store.py` and `agentic_codex/core/observability/logger.py`.
  - How it is used here: the enhanced demo saves a minimal run JSON to `runs/` and the replay cell loads the run to show messages and metadata.
  - Production tips: integrate `RunStore` with an append-only backing store (object storage or DB), and redact PII before persisting. Provide a CLI `agentic replay` (existing CLI references) for operators.

- **Mock MCP tool registration**:
  - Where it's implemented: `agentic_codex/core/tools.py` plus manifests support (`agentic_codex/manifests/*`) for describing remote MCP endpoints and tools.
  - How it is used here: we register a `MockMCPAdapter` and call it from an agent; in production, MCP adapters would call remote servers and expose metadata.
  - Production tips: secure MCP connections with mTLS, add retry/backoff, and ensure rate limits and budgets are applied.

- **Conditional GraphRunner branching**:
  - Where it's implemented: `agentic_codex/core/orchestration` (GraphRunner and GraphNodeSpec usage are shown in examples).
  - How it is used here: base node adds a `security` node into `_graph_additions` based on a low score; GraphRunner runs additional nodes dynamically.



**Comment style & documentation:** each demo cell includes inline comments describing purpose, where to find the core implementation, and how to replace notebook stubs with real integrations (embedding models, DBs, remote MCP endpoints). Use the `examples/` folder as canonical runnable demonstrations when promoting these patterns into CI or docs.  - Production tips: attach guards/validators on `_graph_additions` to avoid unbounded graph growth, and set `branch_budget` to limit expansion.                

In [ ]:
# 1) RAG demo: local index + RAGToolAdapter usage
# Explanation: RAGToolAdapter mimics a lightweight retrieval system. Agents can call it as a tool
# to fetch context which can be injected into prompts for better factual grounding.
from agentic_codex.core.tools import RAGToolAdapter
from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.schemas import AgentStep, Message

# Build a tiny in-memory index mapping short keys to documents
index = {"calc_design": "Calculator must be accurate to 1e-6 and have CLI", "user_guide": "Usage: calc --add --sub --mul --div"}
rag = RAGToolAdapter(index=index)

# Agent that uses rag.query as a tool to fetch context and respond
def rag_step(ctx: Context):
    # Call the RAG tool (available under rag.name)
    tool = ctx.get_tool(rag.name)
    res = tool.invoke(query='calculator', k=2)
    # Create an AgentStep that returns the retrieved results as a message
    return AgentStep(out_messages=[Message(role='rag', content=str(res['results']))], state_updates={'rag_results': res['results']})

rag_agent = AgentBuilder(name='rag_user', role='user').with_capability( 
    # surface the RAG tool under the agent's tools capability
    __import__('agentic_codex').agentic_codex.core.capabilities.ToolCapability({'rag.query': rag})).with_llm(FunctionAdapter(lambda p: 'ok')).with_step(rag_step).build()

# Simpler: register tool directly into context for this quick demo
ctx = Context(goal='rag-demo')
ctx.add_tool(rag.name, rag)
out = rag_agent.run(ctx)
out.out_messages[0].content

# Agentic Components Feature Showcase

This notebook walks through the core features of the Agentic Components framework and ends with a complex, production-style use case. It is heavily annotated so you can adapt it quickly.

### What you'll see
- Lego-style agents (`AgentBuilder`) with capabilities (LLM, tools, memory, prompts, rate limits).
- Prompt management (Prompt-Framework optional) with versioning.
- with_steps (batch/parallel/conditional) and GraphRunner (DAG with dynamic additions).
- CommunicationHub (direct/broadcast/namespace) + rate limits and logging/metrics.
- Memory options (scratch, episodic, semantic, vector) with an in-notebook embedding + retrieval.
- Safe tools (sanitized code exec, safe file read).
- Run persistence (RunStore) and structured logging hooks.
- A complex "Release Readiness" use case tying everything together.

In [ ]:
# Runtime setup: optional OpenAI key with fallback stub LLM
import os
from getpass import getpass

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter OPENAI_API_KEY (press Enter to skip): ')

from agentic_codex import (
    AgentBuilder, Context, FunctionAdapter, EnvOpenAIAdapter, EnvOpenAIEmbeddingAdapter,
    CommunicationHub